# TỔNG KẾT & ĐỐI CHUẨN THỰC NGHIỆM MÔ HÌNH RFM
## So Sánh 3 Thuật Toán Phân Cụm (K-Means, Hierarchical, DBSCAN) & Đối Chiếu Với Baseline

Báo cáo này tổng hợp toàn bộ kết quả thực nghiệm của 3 thuật toán phân cụm trên không gian đặc trưng **RFM (Recency, Frequency, Monetary)**, đưa ra các căn cứ định lượng và thực tiễn để lựa chọn **mô hình RFM tốt nhất**, sau đó tiến hành **đối chiếu chuyên sâu với mô hình Baseline tốt nhất**.

---

### Mục tiêu đối chuẩn:
1. **Tổng hợp số liệu đối chuẩn đa tập:** So sánh hiệu năng của K-Means ($K=4$), Hierarchical ($K=4$, Ward), và DBSCAN ($k=10, \epsilon=0.6$) trên cả 3 tập dữ liệu độc lập: **Train (64%)**, **Validation (16%)**, **Test (20%)**.
2. **So sánh mức độ chi phối đặc trưng (ANOVA F-Score):** Đánh giá mức độ phụ thuộc của từng thuật toán vào các chiều $R$, $F$, $M$.
3. **Phân tích tính ứng dụng kinh doanh:** So sánh khả năng phân khúc khách hàng đại chúng và tách lọc ngoại lai giữa các thuật toán.
4. **Quyết định mô hình RFM tối ưu:** Lựa chọn mô hình chiến thắng trên không gian RFM dựa trên cơ sở khoa học máy học.
5. **So sánh đối đầu RFM Winner vs. Baseline Winner:** Làm sáng tỏ bước nhảy vọt về giá trị nghiệp vụ khi chuyển từ dữ liệu giao dịch thô sang dữ liệu đặc trưng khách hàng RFM.


In [ ]:
import os
import sys

# Tự động định vị thư mục gốc của dự án
current = os.path.abspath(os.getcwd())
while current != os.path.dirname(current):
    if os.path.exists(os.path.join(current, 'data')) and os.path.exists(os.path.join(current, 'Source')):
        break
    current = os.path.dirname(current)

if current not in sys.path:
    sys.path.insert(0, current)
os.chdir(current)
print("Working directory set to:", os.getcwd())

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.neighbors import NearestNeighbors, NearestCentroid
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.feature_selection import f_classif

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Da nap thanh cong toan bo thu vien can thiet!")


### 1. Kiến trúc Thực nghiệm & Nguồn dữ liệu RFM

Quy trình tiền xử lý và phân chia dữ liệu RFM được thực hiện nghiêm ngặt tại `RFM.ipynb` nhằm bảo đảm tính khách quan và chống rò rỉ dữ liệu (Data Leakage):
- **Tập Train (64% - 2,736 khách hàng):** Dùng để huấn luyện mô hình, học tâm cụm và xác lập ranh giới phân vùng.
- **Tập Validation (16% - 684 khách hàng):** Dùng để kiểm định độ ổn định của siêu tham số trên dữ liệu chưa thấy.
- **Tập Test (20% - 855 khách hàng):** Dùng để đánh giá độc lập năng lực tổng quát hóa cuối cùng.
- **Biến đổi dữ liệu:** Áp dụng biến đổi log $\log(x + 1)$ nhằm khử độ lệch phải (Right-Skewness), kết hợp chuẩn hóa `StandardScaler` (fit duy nhất trên tập Train).


In [ ]:
train_path = "data/rfm_train.csv"
val_path = "data/rfm_val.csv"
test_path = "data/rfm_test.csv"

train_rfm = pd.read_csv(train_path)
val_rfm = pd.read_csv(val_path)
test_rfm = pd.read_csv(test_path)

features = ['Recency', 'Frequency', 'Monetary']
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(np.log1p(train_rfm[features]))
X_val_scaled = scaler.transform(np.log1p(val_rfm[features]))
X_test_scaled = scaler.transform(np.log1p(test_rfm[features]))

print(f"Kich thuoc tap Train:      {X_train_scaled.shape[0]:,} khach hang (64%)")
print(f"Kich thuoc tap Validation: {X_val_scaled.shape[0]:,} khach hang (16%)")
print(f"Kich thuoc tap Test:       {X_test_scaled.shape[0]:,} khach hang (20%)")
print(f"Tong so luong khach hang:  {len(train_rfm) + len(val_rfm) + len(test_rfm):,} khach hang")


### 2. Bảng Tổng Hợp Số Liệu Đối Chuẩn 3 Mô Hình RFM trên Train / Validation / Test

Bảng số liệu dưới đây tổng hợp kết quả định lượng của 3 mô hình phân cụm trên cùng một không gian thuộc tính RFM chuẩn hóa qua cả 3 tập dữ liệu:


In [ ]:
benchmark_rfm_data = [
    # K-Means RFM (K=4)
    {"Model": "K-Means (Tuned K=4)", "Dataset": "Train (64%)", "Silhouette": 0.3385, "Davies-Bouldin": 0.9960, "Calinski-Harabasz": 2090.80, "Clusters": 4, "Noise (%)": "0.00%", "Time (s)": "~0.04s"},
    {"Model": "K-Means (Tuned K=4)", "Dataset": "Validation (16%)", "Silhouette": 0.3193, "Davies-Bouldin": 1.0210, "Calinski-Harabasz": 503.80, "Clusters": 4, "Noise (%)": "0.00%", "Time (s)": "~0.01s"},
    {"Model": "K-Means (Tuned K=4)", "Dataset": "Test (20%)", "Silhouette": 0.3253, "Davies-Bouldin": 1.0248, "Calinski-Harabasz": 629.08, "Clusters": 4, "Noise (%)": "0.00%", "Time (s)": "~0.01s"},
    
    # Hierarchical RFM (K=4, Ward)
    {"Model": "Hierarchical (Ward, K=4)", "Dataset": "Train (64%)", "Silhouette": 0.2976, "Davies-Bouldin": 1.0827, "Calinski-Harabasz": 1808.38, "Clusters": 4, "Noise (%)": "0.00%", "Time (s)": "~0.11s"},
    {"Model": "Hierarchical (Ward, K=4)", "Dataset": "Validation (16%)", "Silhouette": 0.3093, "Davies-Bouldin": 1.0281, "Calinski-Harabasz": 485.68, "Clusters": 4, "Noise (%)": "0.00%", "Time (s)": "~0.02s"},
    {"Model": "Hierarchical (Ward, K=4)", "Dataset": "Test (20%)", "Silhouette": 0.3210, "Davies-Bouldin": 1.0485, "Calinski-Harabasz": 621.10, "Clusters": 4, "Noise (%)": "0.00%", "Time (s)": "~0.02s"},
    
    # DBSCAN RFM (eps=0.6, k=10)
    {"Model": "DBSCAN (eps=0.6, k=10)", "Dataset": "Train (64%)", "Silhouette": 0.5254, "Davies-Bouldin": 0.4055, "Calinski-Harabasz": 117.32, "Clusters": 2, "Noise (%)": "1.57%", "Time (s)": "~0.08s"},
    {"Model": "DBSCAN (eps=0.6, k=10)", "Dataset": "Validation (16%)", "Silhouette": 0.5274, "Davies-Bouldin": 0.3382, "Calinski-Harabasz": 17.46, "Clusters": 2, "Noise (%)": "1.46%", "Time (s)": "~0.03s"},
    {"Model": "DBSCAN (eps=0.6, k=10)", "Dataset": "Test (20%)", "Silhouette": 0.5298, "Davies-Bouldin": 0.4017, "Calinski-Harabasz": 25.43, "Clusters": 2, "Noise (%)": "1.64%", "Time (s)": "~0.03s"}
]

df_benchmark = pd.DataFrame(benchmark_rfm_data)

print("==================================================================================================================")
print("                   BANG SO LIEU DOI CHUAN 3 MO HINH PHAN CUM RFM TREN TRAIN / VAL / TEST                        ")
print("==================================================================================================================")
print(df_benchmark.to_string(index=False))


### 3. Trực Quan Hóa So Sánh Hiệu Năng Phân Cụm Giữa 3 Mô Hình

Dưới đây là biểu đồ so sánh trực tiếp các chỉ số **Silhouette Score**, **Davies-Bouldin Index**, và **Calinski-Harabasz Index** trên cả 3 tập dữ liệu nhằm phân tích độ ổn định và tính tổng quát hóa.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = ["K-Means (Tuned K=4)", "Hierarchical (Ward, K=4)", "DBSCAN (eps=0.6, k=10)"]
datasets = ["Train (64%)", "Validation (16%)", "Test (20%)"]
x = np.arange(len(datasets))
width = 0.25

# 1. Silhouette Score Chart
ax1 = axes[0]
sil_km = [0.3385, 0.3193, 0.3253]
sil_h  = [0.2976, 0.3093, 0.3210]
sil_db = [0.5254, 0.5274, 0.5298]

ax1.bar(x - width, sil_km, width, label='K-Means (K=4)', color='#1f77b4', edgecolor='black')
ax1.bar(x, sil_h, width, label='Hierarchical (K=4)', color='#2ca02c', edgecolor='black')
ax1.bar(x + width, sil_db, width, label='DBSCAN (k=10, eps=0.6)*', color='#d62728', edgecolor='black')
ax1.set_title("Silhouette Score (Cang cao cang tot)", fontsize=12, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(datasets)
ax1.set_ylabel("Silhouette Score")
ax1.set_ylim(0, 0.65)
ax1.legend(loc='upper right', frameon=True)
ax1.grid(axis='y', linestyle='--', alpha=0.5)

# 2. Davies-Bouldin Index Chart
ax2 = axes[1]
db_km = [0.9960, 1.0210, 1.0248]
db_h  = [1.0827, 1.0281, 1.0485]
db_db = [0.4055, 0.3382, 0.4017]

ax2.bar(x - width, db_km, width, label='K-Means (K=4)', color='#1f77b4', edgecolor='black')
ax2.bar(x, db_h, width, label='Hierarchical (K=4)', color='#2ca02c', edgecolor='black')
ax2.bar(x + width, db_db, width, label='DBSCAN (k=10, eps=0.6)*', color='#d62728', edgecolor='black')
ax2.set_title("Davies-Bouldin Index (Cang thap cang tot)", fontsize=12, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(datasets)
ax2.set_ylabel("Davies-Bouldin Index")
ax2.set_ylim(0, 1.3)
ax2.legend(loc='upper right', frameon=True)
ax2.grid(axis='y', linestyle='--', alpha=0.5)

# 3. Calinski-Harabasz Index Chart
ax3 = axes[2]
ch_km = [2090.80, 503.80, 629.08]
ch_h  = [1808.38, 485.68, 621.10]
ch_db = [117.32, 17.46, 25.43]

ax3.bar(x - width, ch_km, width, label='K-Means (K=4)', color='#1f77b4', edgecolor='black')
ax3.bar(x, ch_h, width, label='Hierarchical (K=4)', color='#2ca02c', edgecolor='black')
ax3.bar(x + width, ch_db, width, label='DBSCAN (k=10, eps=0.6)', color='#d62728', edgecolor='black')
ax3.set_title("Calinski-Harabasz Index (Cang cao cang tot)", fontsize=12, fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(datasets)
ax3.set_ylabel("Calinski-Harabasz Index")
ax3.set_ylim(0, 2400)
ax3.legend(loc='upper right', frameon=True)
ax3.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


### 4. So Sánh Mức Độ Chi Phối của Các Thuộc Tính RFM (ANOVA F-Score)

Kiểm định thống kê **ANOVA F-Score (`f_classif`)** phản ánh tỷ lệ phương sai giữa các cụm so với phương sai nội cụm trên từng chiều dữ liệu $R, F, M$:
- Thuộc tính có F-Score càng lớn thì càng đóng vai trò then chốt trong việc phân tách các cụm khách hàng.
- Bảng dưới đây so sánh cơ chế phân tách thuộc tính giữa 3 mô hình:


In [ ]:
anova_comparison_data = [
    {
        "Thuoc tinh": "Recency (R)",
        "K-Means RFM (K=4)": "30.50% (F = 1,487.8)",
        "Hierarchical RFM (K=4)": "39.05% (F = 2,165.7)",
        "DBSCAN RFM (eps=0.6)": "56.40% (F = 148.2)"
    },
    {
        "Thuoc tinh": "Frequency (F)",
        "K-Means RFM (K=4)": "39.10% (F = 1,906.7)",
        "Hierarchical RFM (K=4)": "35.60% (F = 1,974.3)",
        "DBSCAN RFM (eps=0.6)": "12.40% (F = 32.6)"
    },
    {
        "Thuoc tinh": "Monetary (M)",
        "K-Means RFM (K=4)": "30.40% (F = 1,482.9)",
        "Hierarchical RFM (K=4)": "25.35% (F = 1,406.1)",
        "DBSCAN RFM (eps=0.6)": "31.20% (F = 81.9)"
    }
]

df_anova = pd.DataFrame(anova_comparison_data)

print("=========================================================================================================")
print("             SO SANH MUC DO DONG GOP (%) CUA CAC THUOC TINH RFM THEO ANOVA F-SCORE                      ")
print("=========================================================================================================")
print(df_anova.to_string(index=False))

# Trực quan hóa so sánh ANOVA
fig, ax = plt.subplots(figsize=(10, 5))
x_anova = np.arange(3)
width_a = 0.25

km_pct = [30.50, 39.10, 30.40]
h_pct  = [39.05, 35.60, 25.35]
db_pct = [56.40, 12.40, 31.20]

ax.bar(x_anova - width_a, km_pct, width_a, label='K-Means RFM', color='#1f77b4', edgecolor='black')
ax.bar(x_anova, h_pct, width_a, label='Hierarchical RFM', color='#2ca02c', edgecolor='black')
ax.bar(x_anova + width_a, db_pct, width_a, label='DBSCAN RFM', color='#d62728', edgecolor='black')

ax.set_title("So sanh Ty le Dong gop (%) cua R, F, M giua 3 Mo hinh", fontsize=13, fontweight='bold')
ax.set_xticks(x_anova)
ax.set_xticklabels(['Recency (R)', 'Frequency (F)', 'Monetary (M)'], fontsize=11)
ax.set_ylabel("Muc do dong gop (%)", fontsize=11)
ax.set_ylim(0, 65)
ax.legend(loc='upper right', frameon=True)
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


### 5. Đánh Giá Giá Trị Kinh Doanh & Khả Năng Hành Động (Business Actionability)

Một mô hình phân cụm học máy trong kinh doanh chỉ thực sự có giá trị khi các cụm tạo ra có khả năng diễn giải và áp dụng được vào các chiến dịch tiếp thị thực tế:

| Tiêu chí | K-Means RFM ($K=4$) | Hierarchical RFM ($K=4$, Ward) | DBSCAN RFM ($k=10, \epsilon=0.6$) |
| :--- | :--- | :--- | :--- |
| **Số lượng phân khúc** | **4 cụm rõ ràng, độc lập** | **4 cụm rõ ràng, độc lập** | **2 cụm mật độ + 1 nhóm Ngoại lai (Noise)** |
| **Độ cân đối tỷ lệ** | Rất cân đối (18% - 33%) | Rất cân đối (18% - 33%) | Mất cân đối trầm trọng (Cụm 0 nuốt 98.4%) |
| **Phân khúc khách hàng đại chúng** | **Xuất sắc:** VIP, Hứa hẹn, Nguy cơ, Rời bỏ | **Xuất sắc:** Tương đồng cao với K-Means | **Kém:** Gom gần như toàn bộ khách hàng vào 1 cụm lớn |
| **Phát hiện khách hàng dị biệt (Outliers)** | Kém hơn (ép ngoại lai vào cụm gần nhất) | Kém hơn (bị gộp vào cây phân cấp) | **Xuất sắc:** Tách lọc chính xác 1.57% nhóm "Cá voi" & Gian lận |
| **Khả năng dự đoán khách hàng mới** | **Cực nhanh:** So khoảng cách tới 4 tâm | **Nhanh:** Qua `NearestCentroid` | Phức tạp: Phải duyệt qua tập mẫu lõi Core Samples |

#### Nhận định chuyên sâu:
1. **DBSCAN:** Điểm Silhouette cao ($0.5254$) của DBSCAN là do thuật toán loại bỏ hoàn toàn các điểm biên/nhiễu và chỉ tính điểm trên các vùng siêu mật độ. Tuy nhiên, việc gom hơn **98% khách hàng vào cụm 0** khiến mô hình này **không thể dùng làm giải pháp phân khúc khách hàng chính**, mà phù hợp nhất với vai trò **bộ lọc ngoại lai / phát hiện gian lận (Anomaly Detection)**.
2. **K-Means & Hierarchical:** Cả hai mô hình đều tạo ra 4 phân khúc có cấu trúc tương đồng và có tính ứng dụng cao, trong đó K-Means đạt chỉ số Calinski-Harabasz và Silhouette nhỉnh hơn, đồng thời giải thuật đơn giản và ổn định hơn.


### 6. Quyết Định Lựa Chọn Mô Hình RFM Tối Ưu Nhất

> ### Mo Hinh Chien Thang Tren Khong Gian RFM: **TUNED K-MEANS CLUSTERING ($K = 4$)**

#### 4 Lý do khoa học và thực tiễn quyết định lựa chọn K-Means ($K=4$):
1. **Hiệu năng định lượng dẫn đầu:** 
   - Điểm **Silhouette Score** trên tập kiểm thử Test đạt **$0.3253$** (vượt qua $0.3210$ của Hierarchical).
   - Chỉ số **Davies-Bouldin Index** trên tập Test đạt mức tối ưu nhất: **$1.0248$** (thấp hơn $1.0485$ của Hierarchical).
   - Chỉ số **Calinski-Harabasz Index** đạt **$2,090.80$** trên Train và **$629.08$** trên Test, vượt trội hoàn toàn so với Hierarchical ($1,808.38$) và DBSCAN ($117.32$).
2. **Cơ chế phân tách thuộc tính đồng đều nhất:**
   - Kết quả ANOVA F-Score cho thấy K-Means khai thác hài hòa cả 3 chiều: $F$ ($39.10\%$), $R$ ($30.50\%$), $M$ ($30.40\%$). Không bị phụ thuộc thái quá vào một chiều đơn lẻ như DBSCAN ($R$ chiếm $56.4\%$).
3. **Ý nghĩa phân khúc khách hàng hoàn hảo:**
   - Tạo ra 4 phân khúc cân đối với chân dung hành vi sắc nét:
     - **Champions (VIP - 20.3%):** $R=14$ ngày, $F=13$ đơn, $M=7,362$ GBP.
     - **Potential Loyalists (Tiềm năng - 32.8%):** $R=22$ ngày, $F=2$ đơn, $M=552$ GBP.
     - **At Risk (Nguy cơ - 26.0%):** $R=86$ ngày, $F=4$ đơn, $M=1,688$ GBP.
     - **Lost (Rời bỏ - 20.9%):** $R=189$ ngày, $F=1.2$ đơn, $M=291$ GBP.
4. **Khả năng triển khai thực tế (Production Readiness):**
   - Lưu trữ trực tiếp tọa độ 4 tâm cụm $3 	imes 4$. Khi có khách hàng mới, việc dự đoán chỉ tốn $O(K 	imes d)$ phép tính khoảng cách Euclid (micro-seconds), dễ dàng tích hợp vào script CLI [predict.py](file:///c:/Users/Quân/Desktop/ml_group/predict.py).


### 7. So Sánh Đối Đầu: Mô Hình RFM Tối Ưu (K-Means K=4) VS. Mô Hình Baseline Tối Ưu (K-Means K=3)

Để khẳng định giá trị của toàn bộ quy trình Trích xuất Đặc trưng (Feature Engineering), bảng dưới đây đối chiếu trực diện mô hình chiến thắng ở cấp độ RFM với mô hình chiến thắng ở cấp độ Baseline:


In [ ]:
comparison_head_to_head = [
    {
        "Tieu chi so sanh": "1. Doi tuong phan tich (Entity Level)",
        "K-Means Baseline (K=3)": "Tung dong hoa don giao dich rieng le",
        "Tuned K-Means RFM (K=4)": "Tung khach hang tong the (CustomerID)"
    },
    {
        "Tieu chi so sanh": "2. Thuoc tinh dau vao (Features)",
        "K-Means Baseline (K=3)": "Quantity, UnitPrice, TotalPrice (Don le)",
        "Tuned K-Means RFM (K=4)": "Recency (ngay), Frequency (so don), Monetary (tong chi tieu)"
    },
    {
        "Tieu chi so sanh": "3. Xu ly phan phoi lech (Skewness)",
        "K-Means Baseline (K=3)": "Khong xu ly log (du lieu bi lech phai cuc nang)",
        "Tuned K-Means RFM (K=4)": "Bien doi log(x+1) dua ve dang tiem can chuan"
    },
    {
        "Tieu chi so sanh": "4. Quy mo du lieu huan luyen",
        "K-Means Baseline (K=3)": "251,501 dong giao dich",
        "Tuned K-Means RFM (K=4)": "2,736 khach hang (Tong hop tu >398,000 giao dich)"
    },
    {
        "Tieu chi so sanh": "5. Silhouette Score tren tap Test",
        "K-Means Baseline (K=3)": "0.6417 (Gia tri ao do lech don bien)",
        "Tuned K-Means RFM (K=4)": "0.3253 (Gia tri thuc chat tren khong gian 3D RFM)"
    },
    {
        "Tieu chi so sanh": "6. Davies-Bouldin tren tap Test",
        "K-Means Baseline (K=3)": "0.6040",
        "Tuned K-Means RFM (K=4)": "1.0248"
    },
    {
        "Tieu chi so sanh": "7. Gia tri ung dung kinh doanh",
        "K-Means Baseline (K=3)": "Rat thap (Chi biet don hang lon hay nho, khong biet ai mua)",
        "Tuned K-Means RFM (K=4)": "Rat cao (Dinh danh chinh xac chan dung, ca nhan hoa tiep thi)"
    }
]

df_h2h = pd.DataFrame(comparison_head_to_head)

print("==================================================================================================================")
print("             DOI CHIEU TRUC DIEN: K-MEANS BASELINE (DATA CLEAN) VS. TUNED K-MEANS RFM (NANG CAO)                  ")
print("==================================================================================================================")
print(df_h2h.to_string(index=False))


#### Giải mã "Nghịch lý Silhouette": Tại sao Baseline đạt 0.6417 nhưng RFM chỉ đạt 0.3253?

Một quan sát trực giác có thể khiến nhiều người băn khoăn:
*Tại sao điểm Silhouette của K-Means Baseline ($0.6417$) lại cao hơn điểm Silhouette của K-Means RFM ($0.3253$)? Phải chăng mô hình Baseline tốt hơn?*

**Câu trả lời dứt khoát là KHÔNG. Đây là hiện tượng phân tách giả do cấu trúc dữ liệu:**
1. **Ở mô hình Baseline:**
   - Dữ liệu đầu vào gồm các giao dịch thô có độ lệch cực lớn: hơn 95% đơn hàng có `Quantity` và `UnitPrice` nhỏ, trong khi một số ít giao dịch có giá trị vượt trội.
   - Khi chạy K-Means, thuật toán dễ dàng cắt dữ liệu thành 3 khối khoảng cách cách xa nhau theo kích thước đơn hàng (đơn hàng siêu nhỏ vs đơn hàng vừa vs đơn siêu lớn). Khoảng cách giữa các khối rất xa nên Silhouette toán học cao ($0.64$), **nhưng hoàn toàn vô nghĩa về mặt kinh doanh** vì không cho biết khách hàng nào là người mua.
2. **Ở mô hình K-Means RFM:**
   - Mỗi điểm dữ liệu là **một con người thực tế (Khách hàng)** với hành vi biến thiên liên tục trong không gian 3 chiều ($R, F, M$). Hành vi tiêu dùng của con người không tồn tại những ranh giới cơ học rõ rệt mà có sự chuyển tiếp mềm mại (ví dụ: ranh giới giữa khách hàng "Sắp rời bỏ" và "Đã rời bỏ").
   - Việc đạt điểm Silhouette **$0.3253$** trên không gian RFM sau khi đã log-transform và chuẩn hóa là **kết quả rất tốt và thực chất**, phản ánh đúng bản chất phân cụm hành vi khách hàng trong các bài toán thương mại điện tử thực tế.

---

### 8. Biểu Đồ Radar Đối Chiếu Chân Dung Phân Cụm Giữa Baseline và RFM

Dưới đây là biểu đồ so sánh trực quan đặc trưng các cụm giữa 2 cấp độ mô hình:


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Subplot 1: Baseline Profiles (Transaction Level)
baseline_clusters = ['Cum 0 (Giao dich pho thong)', 'Cum 1 (Giao dich gia tri cao)', 'Cum 2 (Giao dich so luong lon)']
baseline_shares = [96.5, 2.3, 1.2]
colors_base = ['#7293CB', '#E1974C', '#84BA5B']

ax1.bar(baseline_clusters, baseline_shares, color=colors_base, edgecolor='black', alpha=0.85)
ax1.set_title("K-Means Baseline: Phan bo Giao dich (Transaction-Level)\n(Bi lech: Cum 0 chiem hon 96%)", fontsize=12, fontweight='bold')
ax1.set_ylabel("Ty le phan bo (%)", fontsize=11)
ax1.set_ylim(0, 110)
for i, v in enumerate(baseline_shares):
    ax1.text(i, v + 2, f"{v}%", ha='center', fontweight='bold')
ax1.grid(axis='y', linestyle='--', alpha=0.5)

# Subplot 2: RFM Profiles (Customer Level)
rfm_clusters = ['Champions\n(VIP)', 'Potential\n(Tiem nang)', 'At Risk\n(Nguy co)', 'Lost\n(Roi bo)']
rfm_shares = [20.3, 32.8, 26.0, 20.9]
colors_rfm = ['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728']

ax2.bar(rfm_clusters, rfm_shares, color=colors_rfm, edgecolor='black', alpha=0.85)
ax2.set_title("Tuned K-Means RFM: Phan bo Khach hang (Customer-Level)\n(Can doi ly tuong: 20% - 33% moi cum)", fontsize=12, fontweight='bold')
ax2.set_ylabel("Ty le phan bo (%)", fontsize=11)
ax2.set_ylim(0, 45)
for i, v in enumerate(rfm_shares):
    ax2.text(i, v + 1, f"{v}%", ha='center', fontweight='bold')
ax2.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


---
### 9. Kết Luận Chung & Kiến Nghị Triển Khai Thực Tiễn

#### 1. Kết luận khoa học:
- **Feature Engineering là nhân tố quyết định:** Chuyển đổi dữ liệu từ cấp độ giao dịch đơn lẻ sang cấp độ khách hàng RFM là bước ngoặt cốt lõi, biến một bài toán phân cụm kích thước đơn hàng đơn thuần thành một hệ thống phân khúc khách hàng chiến lược.
- **K-Means ($K=4$) là mô hình tối ưu toàn diện:** Trong cả 3 thuật toán thử nghiệm trên không gian RFM, K-Means ($K=4$) chứng minh tính ưu việt tuyệt đối về cả chỉ số phân cụm (Silhouette $0.3253$, Davies-Bouldin $1.0248$, Calinski-Harabasz $2,090.80$), tính ổn định qua 3 tập Train/Val/Test, và độ cân đối nghiệp vụ.
- **DBSCAN đóng vai trò hỗ trợ đắc lực:** Mặc dù không phù hợp làm mô hình phân khúc đại chúng, DBSCAN ($k=10, \epsilon=0.6$) là công cụ xuất sắc để tách lọc $1.57\%$ khách hàng ngoại lai ("Cá voi" hoặc gian lận), hỗ trợ làm sạch dữ liệu trước khi huấn luyện K-Means.

#### 2. Kiến nghị kiến trúc hệ thống Machine Learning trong thực tế:
Chúng tôi đề xuất mô hình phối hợp **Hybrid Clustering Architecture** cho doanh nghiệp:

```text
[Khach hang moi / Du lieu RFM]
               │
               ▼
   [Buoc 1: DBSCAN Filter] ─── (Khoang cach toi Core > 0.6) ───► [Nhom Sieu VIP / Ngoai lai]
               │                                                  (CSKH 1-on-1 dac quyen)
               ▼ (Diem binh thuong)
   [Buoc 2: K-Means (K=4)] ───► 4 Phan khuc: Champions / Potential / At Risk / Lost
                                (Chien dich Marketing tu dong hoa ca nhan hoa)
```

Kiến trúc kết hợp này tận dụng trọn vẹn điểm mạnh của cả hai thuật toán: DBSCAN bóc tách cá voi và ngoại lai, trong khi K-Means phân khúc chuẩn xác 4 nhóm khách hàng đại chúng!
